In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.ensemble import RandomForestRegressor

sub_ds = load_dataset("CMU-OSPO/protostar-climate-submission-template")

train_ds = load_dataset(
    "CMU-OSPO/protostar-climate-train-to-2018",
    "annual_indices"
)

df = train_ds["train"].to_pandas()
submission = sub_ds["test"].to_pandas()

metric_to_target = {
    "annual_avg_temp": "Avg Temp",
    "total_precip_in": "ToT Prcp",
    "hottest_day_temp": "Max Tmax",
    "cdd": "cdd"
}

df["ID"] = df["ID"].astype(str)
submission["city_id"] = submission["city_id"].astype(str)

df = df.sort_values(["ID", "Year"])
df = df.replace([np.inf, -np.inf], np.nan)

models = {}

for metric, target in metric_to_target.items():

    data = df.copy()

    data["lag_1"] = data.groupby("ID")[target].shift(1)

    features = ["Lat", "Lon", "Year", "lag_1"]

    data = data.dropna(subset=features + [target])

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(data[features], data[target])

    models[metric] = {
        "model": model,
        "target": target
    }

for city_id in submission["city_id"].unique():

    city_history = df[df["ID"] == city_id].copy()
    city_history = city_history.sort_values("Year")

    if city_history.empty:
        continue

    future_years = sorted(
        submission.loc[
            submission["city_id"] == city_id,
            "year"
        ].unique()
    )

    for year in future_years:

        row_predictions = {}

        for metric, info in models.items():

            target = info["target"]
            model = info["model"]

            valid_history = city_history[
                city_history[target].notna()
            ].sort_values("Year")

            if valid_history.empty:
                prediction = np.nan
            else:
                latest = valid_history.iloc[-1]

                X = pd.DataFrame([{
                    "Lat": latest["Lat"],
                    "Lon": latest["Lon"],
                    "Year": year,
                    "lag_1": latest[target]
                }])

                prediction = model.predict(X)[0]

            row_predictions[metric] = prediction

        for metric, prediction in row_predictions.items():

            mask = (
                (submission["city_id"] == city_id)
                & (submission["year"] == year)
                & (submission["metric"] == metric)
            )

            submission.loc[
                mask,
                "predicted_value"
            ] = prediction

        base = city_history.iloc[-1].copy()
        base["ID"] = city_id
        base["Year"] = year

        for metric, prediction in row_predictions.items():
            target = metric_to_target[metric]
            base[target] = prediction

        city_history = pd.concat(
            [
                city_history,
                pd.DataFrame([base])
            ],
            ignore_index=True
        )

submission["predicted_value"] = pd.to_numeric(
    submission["predicted_value"],
    errors="coerce"
)

print("shape:", submission.shape)
print("missing predictions:", submission["predicted_value"].isna().sum())

print(
    submission[submission["predicted_value"].isna()]
)

submission.to_csv("submission.csv", index=False)

Shape: (5880, 4)
Missing predictions: 0
Empty DataFrame
Columns: [city_id, year, metric, predicted_value]
Index: []
